# 程序架构说明

## main
```mermaid
flowchart TD
    Start(["入口: main()"]) --> ParseArgs["解析命令行参数<br/>parse_args()"]
    ParseArgs --> InitLog["初始化双通道日志系统<br/>setup_logging()"]
    
    InitLog --> CheckMaint{"是否触发维护命令?<br/>handle_maintenance()"}
    
    %% 路径 A: 维护分支
    CheckMaint -- 是 --> AssetMgr["实例化 AssetManager"]
    AssetMgr --> ExecMaint["执行 备份/恢复/查询 资产操作"]
    ExecMaint --> ExitMaint(["日志记录并正常退出"])

    %% 路径 B: 天文分析管线分支
    CheckMaint -- 否 --> ValidateCluster["校验/解析目标星团<br/>_validate_cluster()"]
    ValidateCluster --> ParseKV["解析动态微调参数 KEY=VALUE<br/>_parse_key_value_pairs()"]
    
    ParseKV --> InstWorkflow["实例化 AstroWorkflow 管线引擎"]
    InstWorkflow --> RunWorkflow["分发并委托执行管线<br/>wf.run()"]
    
    RunWorkflow --> ExitPipeline(["完成分析流程"])

    %% 异常拦截防护
    subgraph ExceptionHandling ["全局异常防护与安全退出"]
        direction TB
        KeyboardInterrupt["KeyboardInterrupt (Ctrl+C)"] --> LogWarn["警告日志记录并 sys.exit(1)"]
        UnhandledException["未捕获崩溃 Exception"] --> LogCritical["致命错误日志记录并 sys.exit(1)"]
    end

## workflow::run
```mermaid
flowchart TD
    %% -----------------------------------------------------------
    %% 1. PUBLIC API: run() 笛卡尔积调度循环
    %% -----------------------------------------------------------
    subgraph RunBatch ["【入口】 run() 批量调度引擎 (笛卡尔积: Cluster × Category × Mode × Algo)"]
        direction TB
        R1["1. 遍历 Cluster & Category"] --> R2["2. 执行共享数据准备<br/>_prepare_shared_data(ctx_base)"]
        R2 --> R3["3. 遍历 Feature Space & Algorithm"]
        R3 --> R4["4. 构造单次运行上下文 RunContext"]
        R4 --> R5["5. 调用单管线调度器<br/>_execute_single_pipeline(ctx)"]
        R5 --> R6["6. 汇总结果并注册跨星团联合视图<br/>_register_union_view()"]
    end

    %% -----------------------------------------------------------
    %% 2. 核心管线调度: _execute_single_pipeline() (Phase 1 ~ 6)
    %% -----------------------------------------------------------
    subgraph Pipeline6Phase ["【核心调度器】 _execute_single_pipeline() (管线 6 阶段)"]
        direction TB
        
        %% Phase 1
        P1_1["Phase 1: 数据准备 & 环境补全"] --> P1_2["_finalize_context(ctx)"]
        P1_2 --> P2_1

        %% Phase 2
        subgraph P2 ["Phase 2: 成员识别 (_compute_members)"]
            direction TB
            P2_1["数据加载与特征转换<br/>Field / Seeds Transformer"] --> P2_2["初始化 Master 表"]
            P2_2 --> P2_3{使用实验轨道模式<br/>use_experimental?}
            P2_3 -- 否 (老轨) --> P2_4["_run_stable_pipeline()<br/>(PriorGMM)"]
            P2_3 -- 是 (新轨) --> P2_5["_run_experimental_pipeline()<br/>(Core + Spatial Tube 双通道融合)"]
            P2_4 --> P2_6["回灌概率至 Master 表"]
            P2_5 --> P2_6
        end

        P2_6 --> P3_1

        %% Phase 3
        subgraph P3 ["Phase 3: 后处理 (_post_process)"]
            P3_1["打标 is_golden / is_candidate"] --> P3_2["注册候选视图 v_candidates"]
        end

        P3_2 --> P4_1

        %% Phase 4
        subgraph P4 ["Phase 4: 交叉审计 (_audit_phase)"]
            direction TB
            P4_1["1. 交叉比对 _run_cross_match()<br/>(划分 Matched / PG Only / Ref Only)"]
            P4_1 --> P4_2["2. 子集并行审计 _audit_xmatch_subsets()"]
            P4_2 --> P4_3["3. 物理+文献+融合决策 _run_phys_lit_fusion()<br/>(UnifiedMemberValidator)"]
        end

        P4_3 --> P5_1

        %% Phase 5 & 6
        P5_1["Phase 5: 资产导出 (_export_phase)"] --> P6_1["Phase 6: 报告与可视化 (_report_phase)<br/>(AstroAnalyzer.run_phase6)"]
    end

    %% 关联 run() 与 单管线调度器
    R5 --> P1_1
    P6_1 --> R6

## 新轨成员识别
```mermaid
flowchart TD
    Start(["启动 _run_experimental_pipeline"]) --> S1["1. 调度 ClusterSeedExtractor<br/>精炼种子星并回写标签"]
    S1 --> S2["2. 初始化消歧引擎策略<br/>(Bayesian / Threshold / Blind)"]
    S2 --> S3["3. 通道 A: 抓取 Core 核心成员<br/>engine.fit_predict()"]
    
    %% 通道 B 空间管细分
    S3 --> S4["4. 通道 B: 构建 Tidal Tail 空间管"]
    subgraph TubePipeline ["Spatial Tube 潮汐尾捕捉管线"]
        direction TB
        T1["计算自适应管长/管宽<br/>(含 15° Hard Cap 截断)"] --> T2["_build_spatial_tube()<br/>PCA 切面投影 & PM 降级防退化"]
        T2 --> T3["Mahalanobis 运动学 Sigma Clip<br/>(基于高置信 Core 基准)"]
        T3 --> T4["回灌几何标签到 Master 表<br/>(in_tube, pca_long, pca_cross)"]
        T4 --> T5["构建 Tail 种子模板<br/>(排除已知 Core 成员)"]
        T5 --> T6["管内独立两阶段推导<br/>fit() + predict()"]
    end
    
    S4 --> TubePipeline
    TubePipeline --> S5["5. 核心与潮汐尾分层融合 (Hierarchical Union)"]
    
    subgraph UnionLogic ["分层互斥决策逻辑"]
        direction TB
        U1{"core_prob >= 0.5?"}
        U1 -- 是 --> U2["继承 core_prob<br/>标记 source: core / both"]
        U1 -- 否 --> U3{"在管内 且 tail_prob >= 0.5?"}
        U3 -- 是 --> U4["Tail 接管继承 tail_prob<br/>标记 source: tail"]
        U3 -- 否 --> U5["标记 source: field (概率归零)"]
    end
    
    S5 --> UnionLogic
    UnionLogic --> End(["输出最终融合结果 DataFrame"])

## BayesianGmmDisambiguation

```mermaid
flowchart TD
    Start(["开始调用<br/>BayesianGmmDisambiguation"]) --> Choice{调用模式?}

    %% -----------------------------------------------------------
    %% 顶层模式选择路由 (纵向收拢)
    %% -----------------------------------------------------------
    Choice -- "fit()" --> FitProcess
    Choice -- "predict()" --> PredictProcess
    Choice -- "fit_predict()" --> FP_Fit["执行 fit()"] --> FP_Pred["执行 predict()"] --> End

    %% -----------------------------------------------------------
    %% FIT 阶段 (竖向收窄布局)
    %% -----------------------------------------------------------
    subgraph FitProcess ["【Fit 阶段】模型拟合与先验 f 求解"]
        direction TB
        F1["1. 数据清洗 (dropna)<br/>& 特征标准化 (StandardScaler)"] --> F2["2. 拟合背景场 GMM 模型<br/>(支持 subsampling 降采样)"]
        F2 --> F3{启用密度修剪<br/>use_density_prune?}

        %% 修剪分支 (改为纵向单链)
        F3 -- 是 --> F4{修剪算法?}
        F4 -- HDBSCAN --> F5_1["HDBSCAN 聚类<br/>(含 eps=0.0 崩溃降级)"] --> F6
        F4 -- DBSCAN --> F5_2["DBSCAN 聚类"] --> F6

        F6{核心捕获校验}
        F6 -- 成功锁定 --> F7_2["提取核心种子 X_core"]
        F6 -- 过于离散/无凝聚 --> F7_1["回退至全量种子星"]
        F3 -- 否 --> F7_1

        F7_1 --> F8
        F7_2 --> F8["3. 拟合星团核心 GMM 模型"]

        %% EMA 迭代子图 (独立收窄)
        F8 --> F9["4. 计算初始先验 f<br/>与底线保护 f_floor"] --> E1

        subgraph EMALoop ["EMA 极大似然对抗迭代"]
            direction TB
            E1["计算似然 p_cl, p_fi"] --> E2["更新后验概率"] --> E3["更新先验 f_new"]
            E3 --> E4{"Δrel < tol<br/>或达 max_iter?"}
            E4 -- 否 --> E1
        end

        E4 -- 是 (收敛) --> F10["打包返回参数包:<br/>scaler, models, f_converged"]
    end

    %% -----------------------------------------------------------
    %% PREDICT 阶段 (竖向收窄布局)
    %% -----------------------------------------------------------
    subgraph PredictProcess ["【Predict 阶段】全域天区推理"]
        direction TB
        P1["1. 接收目标天区与参数包"] --> P2["2. Scaler 转换特征"]
        P2 --> P3["3. 计算似然 p_cl, p_fi"]
        P3 --> P4["4. 贝叶斯后验推导 (应用 f)"]
        P4 --> P5{存在特征残缺天体?}
        P5 -- 是 --> P6["对齐补全 prob=0.0<br/>野星隔离"] --> P7
        P5 -- 否 --> P7["返回结果 DataFrame<br/>['id', 'prob']"]
    end

    FitProcess --> End
    PredictProcess --> End(["结束，产出结果"])

### Channel A：Core（核心）提取详细流程

```mermaid
flowchart TD
    StartA(["开始: Core 核心提取"]) --> SeedFilter["1. 种子星预处理<br/>dropna & 去重"]
    
    %% 第一阶段：种子星精炼
    subgraph SeedRefine ["第一阶段: ClusterSeedExtractor 种子精炼"]
        direction TB
        SR1["输入原始种子星 df_seeds"] --> SR2{启用 density_prune?}
        SR2 -- 是 --> SR3{选择聚类算法}
        SR3 -- HDBSCAN --> SR4_1["HDBSCAN 密度聚类<br/>(崩溃时自动降级 eps=0.0)"]
        SR3 -- DBSCAN --> SR4_2["DBSCAN 密度聚类"]
        
        SR4_1 --> SR5["校验最大核心类簇样本数 n_best"]
        SR4_2 --> SR5
        
        SR5 --> SR6{"n_best < 20% n_seeds<br/>或全部为野星?"}
        SR6 -- 是 (凝聚失败) --> SR7_1["警告并回退至全量种子星"]
        SR6 -- 否 (锁定成功) --> SR7_2["剔除野星，提取凝聚核心 X_core"]
        SR2 -- 否 --> SR7_1
    end

    SeedFilter --> SeedRefine
    SeedRefine --> TagSeed["2. 回灌标签至 Master 表<br/>seed_type = 'refined_seed'"]

    %% 第二阶段：Core 消歧拟合与推导
    subgraph CoreGMM ["第二阶段: Core 消歧引擎 (bayesian/threshold/blind)"]
        direction TB
        CG1["3. 标准化与背景场拟合<br/>StandardScaler + Field GMM"] --> CG2["4. 星团核心拟合<br/>Cluster GMM (使用精炼种子 X_core)"]
        CG2 --> CG3["5. EMA 贝叶斯对抗递归迭代<br/>求解星团先验密度 f_converged"]
        CG3 --> CG4["6. 全域天区后验推导<br/>应用贝叶斯公式计算 p_cl, p_fi"]
    end

    TagSeed --> CoreGMM
    CoreGMM --> OutputA(["输出 Core 概率结果集<br/>df_res_core (含 prob 列)"])

### Channel B：Tail（潮汐尾）提取详细流程

Tail 提取的目的是**在全天区背景噪声中捕捉沿轨道方向扩散的暗弱潮汐尾天体**。它利用空间管（Spatial Tube）截取局部天区，并通过**剔除已知 Core 成员**构建纯净的 Tail 模板进行二次 GMM 拟合。

```mermaid
flowchart TD
    StartB(["开始: Tail 潮汐尾提取"]) --> CalcDim["1. 计算自适应空间管尺寸<br/>angular_tidal × mult (硬门限截断 <= 15°)"]
    
    %% 第一阶段：空间管几何构建
    subgraph TubeGeo ["第一阶段: _build_spatial_tube 空间管构建"]
        direction TB
        TG1["计算种子星物理质心 (x0, y0)"] --> TG2["转换为正交切面平面坐标 (X_proj, Y_proj)"]
        TG2 --> TG3["拟合 2D PCA 主轴并评估各向异性 ratio"]
        TG3 --> TG4{"ratio < 0.60 且存在自行数据?"}
        TG4 -- 是 (形态退化) --> TG5_1["防退化拦截: 强制切换至<br/>自行矢量 (PM Vector) 引导主轴"]
        TG4 -- 否 (形态良好) --> TG5_2["沿用 2D PCA 主轴"]
        TG5_1 --> TG6["计算自适应半宽 (3σ 横轴散布, 0.8° ~ width_deg)"]
        TG5_2 --> TG6
        TG6 --> TG7["全量天区投影切片<br/>生成 pca_long / pca_cross 几何列"]
    end

    CalcDim --> TubeGeo
    
    %% 第二阶段：运动学剪裁与标签回灌
    subgraph KinematicClip ["第二阶段: 运动学剪裁与数据准备"]
        direction TB
        KC1["从 Core 提取高置信成员 (prob >= 0.5)"] --> KC2{"Pure Core 数量 >= 20?"}
        KC2 -- 是 --> KC3_1["以 Pure Core 5D 特征为基准"]
        KC2 -- 否 --> KC3_2["以 Refined Seeds 为基准 (Fallback)"]
        
        KC3_1 --> KC4["计算管内天体的高维马氏距离 MD"]
        KC3_2 --> KC4
        KC4 --> KC5["σ_clip 过滤野星 (Mahalanobis <= 5.0)"]
    end

    TubeGeo --> KinematicClip
    KinematicClip --> TagTube["2. 回灌几何标签至 Master 表<br/>in_tube=True, pca_long, pca_cross"]

    %% 第三阶段：Tail 模板与管内二次推导
    subgraph TailGMM ["第三阶段: Tail 模板构建与二次消歧推导"]
        direction TB
        TGMM1["取管内天体，筛选 core_prob < 0.2 天体<br/>构建 Tail 种子模板 df_tail_template"] --> TGMM2["管内拟合 (fit):<br/>df_field = df_tube_full<br/>df_seeds = df_tail_template"]
        TGMM2 --> TGMM3["管内推理 (predict):<br/>计算管内全量天体的 tail_prob"]
    end

    TagTube --> TailGMM
    TailGMM --> OutputB(["输出 Tail 概率结果集<br/>df_res_tail (含 prob 列)"])